# Antibiotic Prescribing & Salmonella AMR Resistance Analysis

This notebook investigates the relationship between outpatient antibiotic prescribing rates (CDC, 2015–2022) and antimicrobial resistance (AMR) in clinical *Salmonella* isolates (NCBI Pathogen Detection, 2015–2022) at the US Census-region level. It also explores the role of population density as a potential confounding or contributing factor.

**Research questions:**
- Does higher regional antibiotic prescribing correlate with more AMR in *Salmonella*?
- Is population density independently associated with AMR burden?
- Did COVID-19 (2020) disrupt regional prescribing patterns, and did high-prescribing states recover?

**Key outputs:** `prescribing_vs_resistance.png`, `popdensity_vs_resistance.png`, `resistance_correlations.png`

---
## 1. Setup

Install required packages. `duckdb` is used for fast in-process SQL queries over pandas DataFrames; `pyarrow` enables efficient columnar memory layout.

In [ ]:
!pip -q install duckdb pandas requests pyarrow

---
## 2. CDC Outpatient Antibiotic Prescribing Data (2020–2022)

### 2.1 Verify Data Availability

The CDC publishes state-level outpatient antibiotic prescribing rates as CSV files. We probe each URL before downloading to gracefully handle any unavailable archives. Data for 2020–2022 uses a direct CSV format, while earlier years are embedded in HTML report pages.

In [ ]:
import requests
import pandas as pd

def url_ok(url: str) -> bool:
    try:
        r = requests.get(url, stream=True, timeout=20, allow_redirects=True)
        return r.status_code == 200
    except requests.RequestException:
        return False

# Known-good (from CDC archive pages) for 2020 + 2021
# The archive report pages reference these CSVs. :contentReference[oaicite:3]{index=3}
candidate_urls = {
    # Archive pattern (works for 2019â€“2021)
    2019: "https://archive.cdc.gov/www_cdc_gov/antibiotic-use/data/report-2019_html_files/2019-OAP-Map-States.csv",
    2020: "https://archive.cdc.gov/www_cdc_gov/antibiotic-use/data/report-2020_html_files/2020-OAP-Map-States.csv",
    2021: "https://archive.cdc.gov/www_cdc_gov/antibiotic-use/data/report-2021_html_files/2021-OAP-Map-States.csv",

    # 2022 uses new direct CDC CSV format
    2022: "https://www.cdc.gov/antibiotic-use/media/csv/2022-Outpatient-Rx-Map-Data.csv",
}


available = {y: u for y, u in candidate_urls.items() if url_ok(u)}
missing = {y: u for y, u in candidate_urls.items() if y not in available}

available, missing

### 2.2 Load & Concatenate Available CSVs

Download each available CSV and stack them into a single raw DataFrame, tagging each row with its source year and URL for traceability.

In [ ]:
dfs = []
for year, url in available.items():
    d = pd.read_csv(url)
    d["year"] = year
    d["source_url"] = url
    dfs.append(d)

raw = pd.concat(dfs, ignore_index=True)

raw.shape, raw.head()

### 2.3 Inspect Columns

The CDC changed column schemas between report years. We need to identify which columns carry the rate data so we can unify them in the next step.

In [ ]:
raw.columns.tolist()

### 2.4 Standardize & Clean

Create a unified `rx_per_1000` column by merging the `Rate` field (2020–2021) and the `Prescriptions per 1,000 Population` field (2022). Drop rows with missing values and retain only the three columns we need.

In [ ]:
import numpy as np

clean = raw.copy()

# Standardize column names
clean.columns = [c.strip() for c in clean.columns]

# Create a unified rate column
clean["rx_per_1000"] = np.where(
    clean["Rate"].notna(),
    clean["Rate"],
    clean["Prescriptions per 1,000 Population"]
)

# Convert to numeric
clean["rx_per_1000"] = pd.to_numeric(clean["rx_per_1000"], errors="coerce")

# Standardize state + year
clean["state"] = clean["State"]
clean["year"] = clean["year"].fillna(clean["Year"])

# Keep only what we need
clean = clean[["state", "year", "rx_per_1000"]].dropna()

clean.head(), clean.shape

### 2.5 Register with DuckDB & Verify Coverage

Register the cleaned DataFrame as a DuckDB relation and verify we have all 51 states/DC for each year.

In [ ]:
import duckdb

con = duckdb.connect()
con.register("rx_clean", clean)

In [ ]:
con.execute("""
    SELECT year, COUNT(*) AS n_states
    FROM rx_clean
    GROUP BY year
    ORDER BY year
""").df()

---
## 3. Extending Coverage to 2015–2019

### 3.1 HTML Extraction Function

For 2015–2018, the CDC published data as HTML report pages rather than CSVs. This function first tries `pd.read_html()` to parse structured tables; if that fails it falls back to regex-based text extraction from the raw HTML.

In [ ]:
import pandas as pd
import numpy as np
import re

def extract_state_rates_from_cdc_report(year: int) -> pd.DataFrame:
    """
    Extracts state-level 'antibiotic prescriptions per 1000 persons' rates
    from CDC archive report pages for years where the state list is embedded in HTML.
    Works for 2015-2018 pages like:
      https://archive.cdc.gov/www_cdc_gov/antibiotic-use/data/report-2016.html
    """
    url = f"https://archive.cdc.gov/www_cdc_gov/antibiotic-use/data/report-{year}.html"
    
    # read_html parses tables from the page; some years are not clean tables, so we'll fallback to text parsing
    try:
        tables = pd.read_html(url)
    except ValueError:
        tables = []

    # 1) Try to find a real table that contains State + Rate
    for t in tables:
        cols = [str(c).lower() for c in t.columns]
        if any("state" in c for c in cols) and any("rate" in c for c in cols):
            out = t.copy()
            # normalize column names
            out.columns = [str(c).strip() for c in out.columns]
            # pick likely columns
            state_col = [c for c in out.columns if "state" in c.lower()][0]
            rate_col  = [c for c in out.columns if "rate" in c.lower()][0]
            out = out[[state_col, rate_col]].rename(columns={state_col:"state", rate_col:"rx_per_1000"})
            out["year"] = year
            out["rx_per_1000"] = pd.to_numeric(out["rx_per_1000"], errors="coerce")
            out = out.dropna(subset=["state","rx_per_1000"])
            return out

    # 2) Fallback: parse the inline "State ... Rate" list from page text (works for 2015/2016/2017/2018)
    import requests
    html = requests.get(url, timeout=30).text

    # The state list appears as lines like "Alabama 1,149"
    # We'll capture state names (incl. District of Columbia) + numeric rate
    # This is conservative and will later be filtered to 51 states/DC.
    pattern = re.compile(r"\b([A-Z][A-Za-z]+(?:\s+[A-Z][A-Za-z]+)*)\s+([0-9]{1,3}(?:,[0-9]{3})*|[0-9]+)\b")
    matches = pattern.findall(re.sub(r"<[^>]+>", " ", html))  # strip tags crudely

    # Build candidate dataframe
    cand = pd.DataFrame(matches, columns=["state","rx_per_1000_raw"])
    cand["rx_per_1000"] = (
        cand["rx_per_1000_raw"].str.replace(",","", regex=False)
    )
    cand["rx_per_1000"] = pd.to_numeric(cand["rx_per_1000"], errors="coerce")
    cand["year"] = year

    # Keep only plausible US states/DC by intersecting with the states we already have (from your 2020-2022 CSVs)
    # We'll return raw candidates first; filtering happens in the merge step.
    cand = cand.dropna(subset=["rx_per_1000"])
    return cand[["state","year","rx_per_1000"]]

### 3.2 Install HTML Parser

`lxml` provides a faster and more lenient HTML parser required by `pd.read_html()`.

In [ ]:
!pip install lxml

### 3.3 Scrape 2015–2019 Reports

Loop over years and call the extraction function, then concatenate into `older_raw`.

In [ ]:
older_dfs = []

for y in [2015, 2016, 2017, 2018, 2019]:
    df_y = extract_state_rates_from_cdc_report(y)
    older_dfs.append(df_y)

older_raw = pd.concat(older_dfs, ignore_index=True)

### 3.4 Normalize State Names

Scraped state names contain inconsistent whitespace and casing. This function standardizes them so they merge cleanly with the 2020–2022 data.

In [ ]:
import re

def norm_state(s):
    s = re.sub(r"\s+", " ", str(s)).strip().replace("\xa0", " ")
    s = re.sub(r"\bOf\b", "of", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s.lower() in ["washington dc", "district of columbia", "district of columbia."]:
        return "District of Columbia"

    return s

clean["state"] = clean["state"].apply(norm_state)
older_raw["state"] = older_raw["state"].apply(norm_state)

### 3.5 Filter to Known States

The regex scraper may pick up non-state text. We intersect against the canonical set of states from the 2020–2022 CSVs to remove noise.

In [ ]:
state_set = set(clean["state"].unique())

older = older_raw[older_raw["state"].isin(state_set)].copy()

older.groupby("year")["state"].nunique()

### 3.6 Load 2019 Data from Local CSV

The 2019 archive page was inaccessible at scraping time, so we load from a locally saved CSV (`2019_report.csv`).

In [ ]:
d2019 = pd.read_csv("2019_report.csv")

d2019["state"] = d2019["State"].apply(norm_state)
d2019["year"] = 2019
d2019["rx_per_1000"] = pd.to_numeric(d2019["Rate"], errors="coerce")

d2019 = d2019[["state","year","rx_per_1000"]].dropna()

print("unique states:", d2019["state"].nunique())
d2019.head()

---
## 4. Building the Full State-Year Panel (2015–2022)

### 4.1 Concatenate All Years

Stack the 2015–2018 scraped data, the 2019 CSV, and the 2020–2022 downloads into a single balanced panel (51 states x 8 years = 408 rows).

In [ ]:
panel = pd.concat(
    [
        older[["state","year","rx_per_1000"]],   # 2015â€“2018
        d2019,                                   # 2019
        clean[["state","year","rx_per_1000"]]     # 2020â€“2022
    ],
    ignore_index=True
)

panel["year"] = panel["year"].astype(int)
panel = panel.drop_duplicates()

panel.groupby("year")["state"].nunique().sort_index()

### 4.2 Assign Census Regions

Map each state to one of the four US Census Bureau regions (Northeast, Midwest, South, West) to enable regional aggregation.

In [ ]:
# Census regions mapping
region_map = {
    # Northeast
    "Connecticut":"Northeast","Maine":"Northeast","Massachusetts":"Northeast",
    "New Hampshire":"Northeast","Rhode Island":"Northeast","Vermont":"Northeast",
    "New Jersey":"Northeast","New York":"Northeast","Pennsylvania":"Northeast",

    # Midwest
    "Illinois":"Midwest","Indiana":"Midwest","Michigan":"Midwest","Ohio":"Midwest",
    "Wisconsin":"Midwest","Iowa":"Midwest","Kansas":"Midwest","Minnesota":"Midwest",
    "Missouri":"Midwest","Nebraska":"Midwest","North Dakota":"Midwest",
    "South Dakota":"Midwest",

    # South
    "Delaware":"South","Florida":"South","Georgia":"South","Maryland":"South",
    "North Carolina":"South","South Carolina":"South","Virginia":"South",
    "District of Columbia":"South","West Virginia":"South",
    "Alabama":"South","Kentucky":"South","Mississippi":"South",
    "Tennessee":"South","Arkansas":"South","Louisiana":"South",
    "Oklahoma":"South","Texas":"South",

    # West
    "Arizona":"West","Colorado":"West","Idaho":"West","Montana":"West",
    "Nevada":"West","New Mexico":"West","Utah":"West","Wyoming":"West",
    "Alaska":"West","California":"West","Hawaii":"West",
    "Oregon":"West","Washington":"West"
}

panel["region"] = panel["state"].map(region_map)

panel.groupby("region")["state"].nunique()

### 4.3 Regional Baseline Prescribing Rates (2015–2019)

Compute the pre-COVID baseline average prescribing rate per region. The South is the highest-prescribing region; the West is the lowest.

In [ ]:
import duckdb
con = duckdb.connect()
con.register("panel", panel)

con.execute("""
    SELECT
        region,
        AVG(rx_per_1000) AS baseline_rx
    FROM panel
    WHERE year BETWEEN 2015 AND 2019
    GROUP BY region
    ORDER BY baseline_rx DESC
""").df()

### 4.4 Regional Trends Over Time

In [ ]:
con.execute("""
    SELECT
        region,
        year,
        AVG(rx_per_1000) AS avg_rx
    FROM panel
    GROUP BY region, year
    ORDER BY region, year
""").df()

### 4.5 COVID Impact: Percent Change vs. Baseline

Compare each region's post-COVID prescribing to its 2015–2019 baseline to quantify the disruption caused by the pandemic.

In [ ]:
con.execute("""
WITH baseline AS (
    SELECT region, AVG(rx_per_1000) AS base
    FROM panel
    WHERE year BETWEEN 2015 AND 2019
    GROUP BY region
)
SELECT
    p.region,
    p.year,
    100.0 * (AVG(p.rx_per_1000) - b.base) / b.base AS pct_change
FROM panel p
JOIN baseline b ON p.region = b.region
GROUP BY p.region, p.year, b.base
ORDER BY p.region, p.year
""").df()

---
## 5. Adding Population Density

### 5.1 Fetch 2020 Decennial Census Population

Pull state populations from the US Census Bureau API using the 2020 Decennial Census Public Law 94-171 dataset (`P1_001N` = total population).

In [ ]:
CENSUS_KEY = "185b1ca09f2990cc820a87c2493004bdf478b7fd"

url = (
    f"https://api.census.gov/data/2020/dec/pl"
    f"?get=NAME,P1_001N"
    f"&for=state:*"
    f"&key={CENSUS_KEY}"
)

r = requests.get(url)
data = r.json()

pop = pd.DataFrame(data[1:], columns=data[0])
pop.rename(columns={
    "NAME": "state",
    "P1_001N": "population",
    "state": "state_code"
}, inplace=True)

pop["population"] = pop["population"].astype(int)

pop.head()

### 5.2 Compute Population Density

Divide population by each state's official land area in square miles (US Census Bureau figures) to get persons per square mile.

In [ ]:
# State land area in square miles (US Census Bureau official figures)
land_area_sqmi = {
    "Alabama": 50645, "Alaska": 571951, "Arizona": 113594, "Arkansas": 52035,
    "California": 155779, "Colorado": 103642, "Connecticut": 4842, "Delaware": 1949,
    "District of Columbia": 61, "Florida": 53625, "Georgia": 57513, "Hawaii": 6423,
    "Idaho": 82643, "Illinois": 55519, "Indiana": 35826, "Iowa": 55857,
    "Kansas": 81759, "Kentucky": 39486, "Louisiana": 43204, "Maine": 30843,
    "Maryland": 9707, "Massachusetts": 7800, "Michigan": 56804, "Minnesota": 79627,
    "Mississippi": 46923, "Missouri": 68742, "Montana": 145546, "Nebraska": 76824,
    "Nevada": 109781, "New Hampshire": 8953, "New Jersey": 7354, "New Mexico": 121298,
    "New York": 47126, "North Carolina": 48618, "North Dakota": 69001, "Ohio": 40861,
    "Oklahoma": 68595, "Oregon": 95988, "Pennsylvania": 44743, "Rhode Island": 1034,
    "South Carolina": 30061, "South Dakota": 75811, "Tennessee": 41235, "Texas": 261232,
    "Utah": 82170, "Vermont": 9217, "Virginia": 39490, "Washington": 66456,
    "West Virginia": 24038, "Wisconsin": 54158, "Wyoming": 97093
}

pop["state"] = pop["state"].apply(norm_state)
pop["aland_sqmi"] = pop["state"].map(land_area_sqmi)
pop["pop_density"] = pop["population"] / pop["aland_sqmi"]

pop[["state", "population", "aland_sqmi", "pop_density"]].head()

### 5.3 Merge Density into Panel

In [ ]:
panel = panel.merge(pop[["state", "pop_density"]], on="state", how="left")

---
## 6. Feature Engineering

Add derived features to support downstream modeling and analysis:
- **`rx_yoy`** / **`rx_yoy_pct`**: year-over-year absolute and percent change
- **`state_baseline_2015_2019`**: each state's mean pre-COVID prescribing rate
- **`rx_roll3`**: 3-year rolling mean to smooth year-to-year noise

In [ ]:
panel = panel.sort_values(["state","year"]).copy()

# YoY change (absolute and percent)
panel["rx_yoy"] = panel.groupby("state")["rx_per_1000"].diff(1)
panel["rx_yoy_pct"] = panel.groupby("state")["rx_per_1000"].pct_change(1) * 100

# Baseline (2015â€“2019) per state
baseline = (
    panel.query("2015 <= year <= 2019")
         .groupby("state", as_index=False)["rx_per_1000"]
         .mean()
         .rename(columns={"rx_per_1000":"state_baseline_2015_2019"})
)

panel = panel.merge(baseline, on="state", how="left")

# Rolling 3-year mean (within each state)
panel["rx_roll3"] = (
    panel.groupby("state")["rx_per_1000"]
         .rolling(3, min_periods=1)
         .mean()
         .reset_index(level=0, drop=True)
)

panel.head()

### 6.1 Flag High-Baseline States

Label states in the top quartile of pre-COVID prescribing (>= 75th percentile) as `high_baseline = 1`.

In [ ]:
q75 = baseline["state_baseline_2015_2019"].quantile(0.75)
baseline["high_baseline"] = (baseline["state_baseline_2015_2019"] >= q75).astype(int)
baseline[["state","state_baseline_2015_2019","high_baseline"]].head()

### 6.2 COVID Resilience Score

Compute each state's percentage change in prescribing during 2020 vs. its pre-COVID baseline. States in the top quartile (smallest decline) are flagged `resilient_2020`.

In [ ]:
covid_2020 = panel.query("year == 2020")[["state","rx_per_1000"]].rename(columns={"rx_per_1000":"rx_2020"})
resilience = baseline.merge(covid_2020, on="state", how="left")
resilience["pct_change_2020_vs_base"] = 100*(resilience["rx_2020"] - resilience["state_baseline_2015_2019"]) / resilience["state_baseline_2015_2019"]

# define resilient as top quartile (least negative / most positive change)
q75_res = resilience["pct_change_2020_vs_base"].quantile(0.75)
resilience["resilient_2020"] = (resilience["pct_change_2020_vs_base"] >= q75_res).astype(int)

resilience[["state","pct_change_2020_vs_base","resilient_2020"]].sort_values("pct_change_2020_vs_base", ascending=False).head()

### 6.3 Assemble Modeling Target DataFrame

Merge the baseline features with region and population density for use as model inputs.

In [ ]:
# Choose your target table:
target_df = baseline.merge(panel[["state","region","pop_density"]].drop_duplicates("state"), on="state", how="left")
# If using resilience target:
# target_df = resilience.merge(panel[["state","region","pop_density"]].drop_duplicates("state"), on="state", how="left")

target_df.head()

---
## 7. Panel Data Quality Check

Verify the final panel's shape, dtypes, missing value counts, and year coverage before proceeding to AMR analysis. The 51 missing values in `rx_yoy` are expected (one per state for the first observed year, which has no prior year to diff against).

In [ ]:
print("=== panel shape ===")
print(panel.shape)

print("\n=== columns ===")
print(panel.columns.tolist())

print("\n=== dtypes ===")
print(panel.dtypes)

print("\n=== missing values ===")
print(panel.isnull().sum())

print("\n=== year coverage ===")
print(panel.groupby("year")["state"].nunique().rename("n_states"))

print("\n=== numeric summary ===")
print(panel[["rx_per_1000", "pop_density", "rx_yoy", "rx_yoy_pct", 
             "state_baseline_2015_2019", "rx_roll3"]].describe().round(2))

print("\n=== sample rows (first 10) ===")
panel.sort_values(["state","year"]).head(10)

In [ ]:
print("Shape:", panel.shape)
print("\nColumns:", panel.columns.tolist())
print("\nMissing values:\n", panel.isnull().sum())
print("\nYear coverage:")
print(panel.groupby("year")["state"].nunique().rename("n_states"))
print("\nNumeric summary:")
panel[["rx_per_1000","pop_density","rx_yoy_pct","state_baseline_2015_2019"]].describe().round(2)

---
## 8. NCBI Pathogen Detection: Salmonella AMR Data

### 8.1 Stream & Filter Isolate Metadata

Download the NCBI Pathogen Detection *Salmonella* SNP cluster metadata in chunks directly from the FTP server. We keep only **clinical/human** isolates collected in the **USA between 2015 and 2022**, yielding ~232,000 isolates with AMR gene counts, drug resistance phenotypes, geographic location, and collection year.

In [ ]:
# Better column list now that we know what's available
COLS_NEEDED = [
    "biosample_acc", "collection_date", "geo_loc_name",
    "host", "epi_type", "HHS_region",
    "FDA_lab_id", "collected_by",   
    "isolation_source", "serovar",
    "number_drugs_resistant", "number_drugs_intermediate",
    "number_drugs_susceptible", "number_drugs_tested",
    "number_amr_genes", "number_core_amr_genes",
    "AMR_genotypes", "AMR_genotypes_core", "AST_phenotypes"
]

FTP_URL = "https://ftp.ncbi.nlm.nih.gov/pathogen/Results/Salmonella/latest_snps/Metadata/PDG000000002.3928.metadata.tsv"

print("Streaming with updated columns...")
chunks = []

chunk_iter = pd.read_csv(
    FTP_URL,
    sep="\t",
    usecols=lambda c: c in COLS_NEEDED,
    chunksize=50_000,
    low_memory=False
)

for i, chunk in enumerate(chunk_iter):
    chunk["country"] = chunk["geo_loc_name"].astype(str).str.split(":").str[0].str.strip()
    chunk["state_raw"] = chunk["geo_loc_name"].astype(str).str.split(":").str[-1].str.strip()
    chunk["year"] = pd.to_numeric(
        chunk["collection_date"].astype(str).str[:4], errors="coerce"
    )
    filtered = chunk[
        (chunk["country"] == "USA") &
        (chunk["year"] >= 2015) &
        (chunk["year"] <= 2022) &
        (chunk["epi_type"].astype(str).str.lower().isin(["clinical", "human"]))
    ]
    if len(filtered) > 0:
        chunks.append(filtered)

    if i % 10 == 0:
        print(f"  {i * 50_000:,} rows processed, {sum(len(c) for c in chunks):,} kept...")

df_sal = pd.concat(chunks, ignore_index=True)
print(f"\nFinal: {len(df_sal):,} clinical USA Salmonella isolates (2015â€“2022)")

### 8.2 Explore Geographic Coverage Fields

Assess how well-populated the geographic fields are. The majority of isolates report only `USA` in `geo_loc_name` with no state; we therefore use the `collected_by` institution name and the `HHS_region` numeric code as geographic proxies.

In [ ]:
for col in ["FDA_lab_id", "HHS_region", "collected_by", "geo_loc_name"]:
    print(f"\n=== {col} ===")
    print(f"Non-null: {df_sal[col].notna().sum():,} / {len(df_sal):,}")
    print(df_sal[col].value_counts().head(15))

### 8.3 Map Isolates to Census Regions

Apply a prioritized lookup chain to assign each isolate to a Census region:
1. `HHS_region` code (most reliable)
2. Known institution name -> state -> Census region
3. Exact state abbreviation in `collected_by`
4. State name substring in `collected_by`
5. State parsed from `geo_loc_name`

> **Note:** Only ~10% of isolates can be mapped to a region. The remainder list only `CDC` as collector with no further geographic detail.

In [ ]:
# Institution to state lookup for known facilities
institution_state_map = {
    "cornell university": "New York",
    "brigham and women": "Massachusetts",
    "nphl": "Nebraska",
    "contra costa public health": "California",
    "penn state animal diagnostic laboratory": "Pennsylvania",
    "tgen north": "Arizona",
    "texas a&m university": "Texas",
    "minnesota veterinary diagnostic laboratory": "Minnesota",
    "united states food and drug administration": None,  # no state info
    "ucsf": "California",                               # UC San Francisco
    "cdc:wy": "Wyoming",                                # CDC:WY -> Wyoming
    "marcos henrique de moraes": None,                  
    "missimg": None,                                    # typo for "missing"
}

state_patterns = {
    "Minnesota": "Minnesota",
    "New York": "New York",
    "Maryland": "Maryland",
    "Texas": "Texas",
    "Arizona": "Arizona",
    "Pennsylvania": "Pennsylvania",
    "South Carolina": "South Carolina",
    "Hawaii": "Hawaii",
    "Oregon": "Oregon",
    "Michigan": "Michigan",
    "Florida": "Florida",
    "Wisconsin": "Wisconsin",
    "Colorado": "Colorado",
    "California": "California",
    "Nebraska": "Nebraska",
    "Wyoming": "Wyoming",
    "Utah": "Utah",
    "Washington": "Washington",
    "Idaho": "Idaho",
    "Montana": "Montana",
    "North Dakota": "North Dakota",
    "South Dakota": "South Dakota",
    "Kansas": "Kansas",
    "Iowa": "Iowa",
    "Missouri": "Missouri",
    "Illinois": "Illinois",
    "Indiana": "Indiana",
    "Ohio": "Ohio",
    "Tennessee": "Tennessee",
    "Georgia": "Georgia",
    "Alabama": "Alabama",
    "Mississippi": "Mississippi",
    "Louisiana": "Louisiana",
    "Arkansas": "Arkansas",
    "Oklahoma": "Oklahoma",
    "Virginia": "Virginia",
    "West Virginia": "West Virginia",
    "North Carolina": "North Carolina",
    "Delaware": "Delaware",
    "Connecticut": "Connecticut",
    "Massachusetts": "Massachusetts",
    "Rhode Island": "Rhode Island",
    "Vermont": "Vermont",
    "Maine": "Maine",
    "New Hampshire": "New Hampshire",
    "New Jersey": "New Jersey",
    "New Mexico": "New Mexico",
    "Nevada": "Nevada",
    "Alaska": "Alaska",
    "District of Columbia": "District of Columbia",
}

abbrev_map = {
    "AL":"Alabama","AK":"Alaska","AZ":"Arizona","AR":"Arkansas","CA":"California",
    "CO":"Colorado","CT":"Connecticut","DE":"Delaware","FL":"Florida","GA":"Georgia",
    "HI":"Hawaii","ID":"Idaho","IL":"Illinois","IN":"Indiana","IA":"Iowa",
    "KS":"Kansas","KY":"Kentucky","LA":"Louisiana","ME":"Maine","MD":"Maryland",
    "MA":"Massachusetts","MI":"Michigan","MN":"Minnesota","MS":"Mississippi",
    "MO":"Missouri","MT":"Montana","NE":"Nebraska","NV":"Nevada","NH":"New Hampshire",
    "NJ":"New Jersey","NM":"New Mexico","NY":"New York","NC":"North Carolina",
    "ND":"North Dakota","OH":"Ohio","OK":"Oklahoma","OR":"Oregon","PA":"Pennsylvania",
    "RI":"Rhode Island","SC":"South Carolina","SD":"South Dakota","TN":"Tennessee",
    "TX":"Texas","UT":"Utah","VT":"Vermont","VA":"Virginia","WA":"Washington",
    "WV":"West Virginia","WI":"Wisconsin","WY":"Wyoming","DC":"District of Columbia"
}

hhs_to_census = {
    1.0: "Northeast", 2.0: "Northeast",
    3.0: "South",     4.0: "South",    6.0: "South",
    5.0: "Midwest",   7.0: "Midwest",
    8.0: "West",      9.0: "West",     10.0: "West",
}

def extract_region(row):
    # Priority 1: HHS region (most reliable)
    if pd.notna(row["HHS_region"]):
        return hhs_to_census.get(row["HHS_region"])

    cb = str(row["collected_by"]).strip().rstrip("_")
    cb_lower = cb.lower()

    # Priority 2: known institution lookup  <-- NEW
    for institution, state in institution_state_map.items():
        if institution in cb_lower:
            if state is None:
                return None
            return region_map.get(state)

    # Priority 3: exact abbreviation match
    if cb in abbrev_map:
        return region_map.get(abbrev_map[cb])

    # Priority 4: state name substring
    for state_name in state_patterns:
        if state_name.lower() in cb_lower:
            return region_map.get(state_name)

    # Priority 5: geo_loc_name state parsing
    geo = str(row["geo_loc_name"])
    if ":" in geo:
        state_part = geo.split(":")[-1].strip()
        if state_part in region_map:
            return region_map[state_part]
        if state_part in abbrev_map:
            return region_map.get(abbrev_map[state_part])

    return None

print("Extracting regions (this may take a moment on 232k rows)...")
df_sal["region"] = df_sal.apply(extract_region, axis=1)

print("\n=== Region coverage ===")
print(df_sal["region"].value_counts())
print(f"\nMapped:   {df_sal['region'].notna().sum():,} / {len(df_sal):,}")
print(f"Unmapped: {df_sal['region'].isna().sum():,} / {len(df_sal):,}")

unmapped = df_sal[df_sal["region"].isna()]["collected_by"].value_counts().head(25)
print("\n=== Top unmapped collected_by values ===")
print(unmapped)

### 8.4 Derive AMR Indicators & Aggregate to Region-Year

- **`has_core_amr`**: isolate carries at least one *core* AMR gene (clinically acquired; excludes intrinsic *mdsA/mdsB* efflux genes)
- **`has_any_amr`**: isolate has >2 total AMR genes (broad indicator)

Aggregate to region-year level to compute `pct_core_amr`, `pct_any_amr`, and `avg_amr_genes`.

In [ ]:
df_sal_region = df_sal[df_sal["region"].notna()].copy()

# Core AMR = clinically acquired resistance genes (excludes native mdsA/mdsB)
df_sal_region["has_core_amr"] = (df_sal_region["number_core_amr_genes"] > 0).astype(int)
df_sal_region["has_any_amr"]  = (df_sal_region["number_amr_genes"] > 2).astype(int)

region_resistance = (
    df_sal_region.groupby(["region", "year"])
    .agg(
        n_isolates        = ("biosample_acc", "count"),
        avg_amr_genes     = ("number_amr_genes", "mean"),
        avg_core_amr_genes= ("number_core_amr_genes", "mean"),
        pct_core_amr      = ("has_core_amr", "mean"),
        pct_any_amr       = ("has_any_amr", "mean"),
    )
    .reset_index()
)

region_resistance["pct_core_amr"] = (region_resistance["pct_core_amr"] * 100).round(2)
region_resistance["pct_any_amr"]  = (region_resistance["pct_any_amr"]  * 100).round(2)
region_resistance["year"]         = region_resistance["year"].astype(int)

print("=== Resistance panel ===")
print(region_resistance.to_string(index=False))

print("\n=== Isolates per region per year ===")
print(region_resistance.pivot(
    index="year", columns="region", values="n_isolates"
).fillna(0).astype(int))

---
## 9. Merging AMR Data with Prescribing Panel

### 9.1 Join on Region x Year

Aggregate the prescribing panel to region-year means (avg prescribing rate, avg population density, avg baseline) and merge with the Salmonella AMR summary.

In [ ]:
# Get region-year avg prescribing rate from your existing panel
rx_region = (
    panel.groupby(["region", "year"], as_index=False)
    .agg(
        avg_rx_per_1000       = ("rx_per_1000", "mean"),
        avg_pop_density       = ("pop_density", "mean"),
        avg_baseline_rx       = ("state_baseline_2015_2019", "mean"),
    )
)

# Merge
merged = rx_region.merge(region_resistance, on=["region", "year"], how="inner")

print("=== Merged panel shape:", merged.shape, "===")
print(f"Years: {sorted(merged['year'].unique())}")
print(f"Regions: {sorted(merged['region'].unique())}")
print()
print(merged.to_string(index=False))

### 9.2 Flag Sparse Observations

Region-years with fewer than 100 isolates have unreliable AMR estimates (high sampling variance). We flag these as `low_n` and create a `merged_reliable` subset for statistical inference.

In [ ]:
# Flag sparse rows - use n < 100 as threshold
merged["low_n"] = merged["n_isolates"] < 100

print("=== Sparse rows (n < 100 isolates) ===")
print(merged[merged["low_n"]][["region", "year", "n_isolates", "pct_core_amr"]].to_string(index=False))

# Two analysis subsets
merged_full    = merged.copy()                        # all 32 rows
merged_reliable = merged[~merged["low_n"]].copy()    # 20 rows, n >= 100 only

print(f"\nFull panel:     {len(merged_full)} region-year observations")
print(f"Reliable panel: {len(merged_reliable)} region-year observations")
print(f"Dropped rows:   {merged['low_n'].sum()}")

---
## 10. Visualizations

### 10.1 Four-Panel Correlation Overview (`resistance_correlations.png`)

Scatter plots of all pairwise combinations of prescribing rate, population density, and AMR outcomes. Color-coded by Census region; OLS trend lines included.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()

comparisons = [
    ("avg_rx_per_1000", "pct_core_amr",
     "Avg Rx per 1,000", "% Isolates with Core AMR Genes",
     "Prescribing Rate vs. Core AMR Resistance"),
    ("avg_rx_per_1000", "pct_any_amr",
     "Avg Rx per 1,000", "% Isolates with Any AMR Genes",
     "Prescribing Rate vs. Any AMR Resistance"),
    ("avg_pop_density", "pct_core_amr",
     "Avg Population Density (per sq mi)", "% Isolates with Core AMR Genes",
     "Population Density vs. Core AMR Resistance"),
    ("avg_pop_density", "avg_amr_genes",
     "Avg Population Density (per sq mi)", "Avg AMR Gene Count",
     "Population Density vs. Avg AMR Gene Count"),
]

colors = {"Northeast": "steelblue", "South": "tomato",
          "Midwest": "seagreen", "West": "darkorange"}

for ax, (x, y, xlabel, ylabel, title) in zip(axes, comparisons):
    for region, grp in merged_reliable.groupby("region"):
        ax.scatter(grp[x], grp[y], label=region,
                   color=colors[region], alpha=0.7, s=60)

    # Overall trend line
    r, p = stats.pearsonr(merged_reliable[x], merged_reliable[y])
    m, b = np.polyfit(merged_reliable[x], merged_reliable[y], 1)
    xline = np.linspace(merged_reliable[x].min(), merged_reliable[x].max(), 100)
    ax.plot(xline, m * xline + b, color="black", linestyle="--", linewidth=1.2)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title}\nr = {r:.2f}, p = {p:.3f}")
    ax.legend(fontsize=8)

plt.suptitle("Antibiotic Prescribing, Population Density & Resistance\n(Regional, 2015â€“2022)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("resistance_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

### 10.2 Regression Modeling (`prescribing_vs_resistance.png`)

OLS and regularized regression (Ridge) with leave-one-group-out cross-validation using region as the group variable. Tests whether prescribing rate or population density better predicts AMR after controlling for temporal trends.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

model_df = merged_reliable.dropna(subset=["pct_core_amr", "avg_rx_per_1000",
                                  "avg_pop_density", "year"]).copy()

model_df["region_enc"] = LabelEncoder().fit_transform(model_df["region"])

# Features: prescribing rate, pop density, region, year
X = model_df[["avg_rx_per_1000", "avg_pop_density", "region_enc", "year"]]
y = model_df["pct_core_amr"]
groups = model_df["region_enc"]  # for group-aware CV

# Leave-one-region-out CV - more rigorous than random kfold for regional panel data
logo = LeaveOneGroupOut()

models = {
    "Linear Regression" : Pipeline([("scaler", StandardScaler()),
                                     ("model", LinearRegression())]),
    "Ridge Regression"  : Pipeline([("scaler", StandardScaler()),
                                     ("model", Ridge(alpha=1.0))]),
    "Random Forest"     : RandomForestRegressor(n_estimators=200, random_state=42),
    "Gradient Boosting" : GradientBoostingRegressor(n_estimators=200, random_state=42),
}

print("=== Model performance (Leave-One-Region-Out CV) ===")
print(f"{'Model':<22} {'RÂ² mean':>10} {'RÂ² std':>10} {'RMSE mean':>12}")
print("-" * 56)

results = {}
for name, model in models.items():
    r2    = cross_val_score(model, X, y, cv=logo, groups=groups, scoring="r2")
    rmse  = np.sqrt(-cross_val_score(model, X, y, cv=logo, groups=groups,
                                      scoring="neg_mean_squared_error"))
    results[name] = {"r2_mean": r2.mean(), "r2_std": r2.std(), "rmse_mean": rmse.mean()}
    print(f"{name:<22} {r2.mean():>10.3f} {r2.std():>10.3f} {rmse.mean():>12.3f}")

# Feature importance from best tree model
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X, y)
feat_imp = pd.Series(rf.feature_importances_,
                     index=X.columns).sort_values(ascending=False)
print("\n=== Random Forest Feature Importances ===")
print(feat_imp.round(3))

import statsmodels.formula.api as smf

print("=== OLS Regression Summary ===")
ols = smf.ols(
    "pct_core_amr ~ avg_rx_per_1000 + avg_pop_density + C(region) + year",
    data=merged_reliable
).fit()
print(ols.summary())

### 10.3 Correlation Matrix & Bivariate Tests

Pearson correlations among the key predictors and the AMR outcome, followed by bivariate OLS. Year is excluded from multivariate models due to collinearity with the prescribing rate trend.

In [ ]:
# Drop year from OLS since it's collinear with rx rate
# Use prescribing rate change instead of level, or just bivariate

print("=== Correlation matrix ===")
print(merged_reliable[["avg_rx_per_1000", "year", "avg_pop_density", 
                         "pct_core_amr"]].corr().round(3))

# Bivariate: prescribing rate vs resistance, by region
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

fig, ax = plt.subplots(figsize=(9, 6))
colors = {"Northeast": "steelblue", "South": "tomato",
          "Midwest": "seagreen", "West": "darkorange"}

for region, grp in merged_reliable.groupby("region"):
    sc = ax.scatter(grp["avg_rx_per_1000"], grp["pct_core_amr"],
                    label=region, color=colors[region], s=80, alpha=0.85)
    # Annotate year
    for _, row in grp.iterrows():
        ax.annotate(str(row["year"])[-2:],
                    (row["avg_rx_per_1000"], row["pct_core_amr"]),
                    textcoords="offset points", xytext=(5, 3), fontsize=7,
                    color=colors[region])

# Overall trend
r, p = stats.pearsonr(merged_reliable["avg_rx_per_1000"],
                       merged_reliable["pct_core_amr"])
m, b = np.polyfit(merged_reliable["avg_rx_per_1000"],
                   merged_reliable["pct_core_amr"], 1)
xline = np.linspace(merged_reliable["avg_rx_per_1000"].min(),
                     merged_reliable["avg_rx_per_1000"].max(), 100)
ax.plot(xline, m * xline + b, "k--", linewidth=1.2,
        label=f"Overall trend (r={r:.2f}, p={p:.3f})")

ax.set_xlabel("Avg Regional Antibiotic Prescribing Rate (Rx per 1,000)", fontsize=11)
ax.set_ylabel("% Salmonella Isolates with Core AMR Genes", fontsize=11)
ax.set_title("Antibiotic Prescribing Rate vs. Salmonella AMR Resistance\nby Census Region, 2015â€“2022 (n≥100 isolates)", fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig("prescribing_vs_resistance.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nPearson r = {r:.3f}, p = {p:.3f}")

### 10.4 Population Density vs. Salmonella AMR Resistance (`popdensity_vs_resistance.png`)

Scatter of average regional population density vs. percentage of isolates carrying core AMR genes, by region and year. Points are labeled with 2-digit year suffixes to reveal temporal trajectories within each region.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

for region, grp in merged_reliable.groupby("region"):
    ax.scatter(grp["avg_pop_density"], grp["pct_core_amr"],
               label=region, color=colors[region], s=80, alpha=0.85)
    for _, row in grp.iterrows():
        ax.annotate(str(row["year"])[-2:],
                    (row["avg_pop_density"], row["pct_core_amr"]),
                    textcoords="offset points", xytext=(5, 3), fontsize=7,
                    color=colors[region])

r2, p2 = stats.pearsonr(merged_reliable["avg_pop_density"],
                          merged_reliable["pct_core_amr"])
m2, b2 = np.polyfit(merged_reliable["avg_pop_density"],
                     merged_reliable["pct_core_amr"], 1)
xline2 = np.linspace(merged_reliable["avg_pop_density"].min(),
                      merged_reliable["avg_pop_density"].max(), 100)
ax.plot(xline2, m2 * xline2 + b2, "k--", linewidth=1.2,
        label=f"Overall trend (r={r2:.2f}, p={p2:.3f})")

ax.set_xlabel("Avg Population Density (persons per sq mi)", fontsize=11)
ax.set_ylabel("% Salmonella Isolates with Core AMR Genes", fontsize=11)
ax.set_title("Population Density vs. Salmonella AMR Resistance\nby Census Region, 2015â€“2022 (n≥100 isolates)", fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig("popdensity_vs_resistance.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Pearson r = {r2:.3f}, p = {p2:.3f}")

---
## 11. Within-Region Correlation Analysis

Test whether AMR increases over time and whether it tracks the prescribing rate *within* each region separately. This controls for between-region confounders such as baseline demographics and healthcare access patterns.

In [ ]:
print("=== Within-region correlation: year vs pct_core_amr ===")
for region, grp in merged_reliable.groupby("region"):
    r_yr, p_yr = stats.pearsonr(grp["year"], grp["pct_core_amr"])
    r_rx, p_rx = stats.pearsonr(grp["avg_rx_per_1000"], grp["pct_core_amr"])
    print(f"{region:<12}  year r={r_yr:+.2f} (p={p_yr:.3f})   rx_rate r={r_rx:+.2f} (p={p_rx:.3f})   n={len(grp)}")

---
## 12. National vs. Regional AMR Comparison

Compare the region-panel weighted average AMR rate to the national aggregate (all isolates including those that could not be mapped to a region). Close agreement validates that the ~10% mapped subset is broadly representative of the national trend.

In [ ]:
# National baseline from all isolates (including unmapped)
national = (
    df_sal_region.groupby("year")  # use full df_sal here, not just regional
    .agg(
        n_isolates=("biosample_acc", "count"),
        pct_core_amr=("has_core_amr", "mean"),
    )
    .reset_index()
)
national["pct_core_amr"] = (national["pct_core_amr"] * 100).round(2)

# Compare to your regional panel weighted average
regional_weighted = (
    merged_reliable.groupby("year")
    .apply(lambda x: np.average(x["pct_core_amr"], weights=x["n_isolates"]))
    .reset_index(name="weighted_pct_core_amr")
)

print("=== National (all isolates) vs Regional panel weighted avg ===")
comparison = national.merge(regional_weighted, on="year", how="inner")
print(comparison.to_string(index=False))

---
## 13. Summary & Limitations

### Key Findings
- **Overall (cross-region):** A moderate negative correlation (r = -0.55, p = 0.014) between population density and core AMR prevalence suggests denser, lower-prescribing regions (Northeast/West) do *not* have higher AMR in *Salmonella*.
- **Within-region (Midwest/Northeast):** AMR rose significantly over time (r > 0.93) while prescribing fell, implying temporal AMR trends are not driven by local antibiotic use.
- **COVID disruption:** All regions saw sharp prescribing declines in 2020, with partial recovery by 2022. Southern states were most resilient (smallest drop).

### Limitations
- Only ~10% of Salmonella isolates could be mapped to a Census region, limiting statistical power.
- AMR gene presence (genotype) does not perfectly predict clinical phenotypic resistance.
- Ecological correlations (region-level) cannot establish individual-level causation.
- Population density from the 2020 Census is applied uniformly across all years.